**[🏠 Course Home](00_START_HERE.ipynb) | 🔵 R Track | 📊 Sheet: Estimating a Mean | [📊 Next Sheet: Two-Sample Comparison (t-Test)](sheet_t_test_frequentist_vs_bayesian.ipynb)**

---

# Sheet: Calculating a Mean in a Frequentist World
### *Point Estimation, Sampling Distributions, Standard Errors, Central Limit Theorem, and the Anatomy of Confidence Intervals*

* **Authoritative Track Reference**: [🐍 Python Lab Companion](../python/sheet_calculating_a_mean_in_a_frequentist_world.ipynb)
* **Conceptual Companion Guide**: [📖 Conceptual Guide: The 100 Universes](../conceptual/sheet_calculating_a_mean_in_a_frequentist_world.ipynb)



## 🧭 Executive Summary: The Frequentist Estimation Engine at a Glance

| Concept | Mathematical Symbol / Formula | Frequentist Meaning | Critical Pitfall / Misconception to Avoid |
| :--- | :--- | :--- | :--- |
| **True Population Mean** | $\mu = \mathbb{E}[X] = \int x f(x) dx$ | Fixed, immutable constant of nature. It has **no probability distribution**. | Thinking $\mu$ is a random variable that can "fall into" intervals. |
| **Sample Mean Estimator** | $\bar{X} = \frac{1}{n}\sum_{i=1}^n X_i$ | Random variable across repeated samples; MVUE and MLE under normality. | Treating $\bar{x}$ as "the true mean" rather than a single noisy draw. |
| **Sample Standard Deviation** | $s = \sqrt{\frac{1}{n-1}\sum (X_i - \bar{X})^2}$ | Measures **variability of individual observations**. Uses Bessel's correction ($n-1$). | Confusing with standard error; $s$ **does not shrink** as $n \to \infty$. |
| **Standard Error of the Mean** | $SE(\bar{X}) = \frac{\sigma}{\sqrt{n}} \approx \frac{s}{\sqrt{n}}$ | Measures **uncertainty of the estimator** across hypothetical repetitions. | Confusing with $s$; $SE$ shrinks at rate $1/\sqrt{n}$ as $n \to \infty$. |
| **Sampling Distribution** | $\bar{X} \sim \mathcal{N}\left(\mu, \frac{\sigma^2}{n}\right)$ or $\frac{\bar{X}-\mu}{s/\sqrt{n}} \sim t_{n-1}$ | The probability distribution of $\bar{X}$ if the experiment were repeated infinitely. | Confusing the distribution of the data with the distribution of $\bar{X}$. |
| **95% Confidence Interval** | $\left[\bar{x} - t_{0.025, n-1}\frac{s}{\sqrt{n}}, \; \bar{x} + t_{0.025, n-1}\frac{s}{\sqrt{n}}\right]$ | In 95% of repeated identical experiments, the calculated interval covers $\mu$. | Claiming: *"There is a 95% chance $\mu$ is in $[L, U]$"* (that is Bayesian). |
| **Margin of Error ($E$)** | $E = t^* \cdot \frac{s}{\sqrt{n}}$ | The half-width of the confidence interval at a specified confidence level. | Assuming doubling $n$ halves $E$; quadrupling $n$ is required. |
| **Bootstrap Mean & CI** | $\bar{X}^* \sim \text{Resample}(X, n)$ | Non-parametric estimate of sampling distribution without assuming normality. | Still subject to small-sample bias if $n$ is too tiny to represent tails. |
| **Trimmed Mean** | $\bar{x}_{\alpha} = \frac{1}{n - 2k}\sum_{i=k+1}^{n-k} X_{(i)}$ | Discards the top and bottom $\alpha\%$ of sorted observations; high breakdown point. | Loses efficiency if data is genuinely Gaussian without outliers. |



## Part 1: The Production Scenario & Observed Data

### 1.1 The Engineering Context
Suppose we are site reliability engineers monitoring an asynchronous transaction processing pipeline. We want to estimate the **mean processing duration (latency in milliseconds)** of this pipeline.

In the real world:
* Nature has a true Data Generating Process (DGP) with true population mean $\mu$ and standard deviation $\sigma$.
* We cannot inspect infinite transactions; we pull a single production telemetry sample of $n = 35$ consecutive transaction durations.
* We must compute the point estimate and rigorously quantify our uncertainty using frequentist statistical principles.

Let's generate synthetic benchmark data from a known ground truth to walk through the entire frequentist workflow.



In [ ]:
# Set random seed for exact reproducibility
set.seed(42)

# Ground truth population parameters (hidden from engineer)
true_mu <- 145.0   # True mean latency (ms)
true_sigma <- 15.0 # True population standard deviation (ms)
n <- 50            # Sample size (observed requests)

# Generate synthetic latency measurements from Normal distribution
sample_data <- rnorm(n, mean = true_mu, sd = true_sigma)

# Compute core sample statistics
x_bar <- mean(sample_data)
s_unbiased <- sd(sample_data)
se_mean <- s_unbiased / sqrt(n)

cat("==================================================
")
cat("     SAMPLE DATASET & FREQUENTIST SUMMARY         
")
cat("==================================================
")
cat(sprintf("Sample Size (n)                   : %d requests
", n))
cat(sprintf("Sample Mean (x_bar)               : %.3f ms
", x_bar))
cat(sprintf("Sample Std Dev (s, ddof=1)        : %.3f ms
", s_unbiased))
cat(sprintf("Standard Error of Mean (SE)       : %.3f ms
", se_mean))
cat(sprintf("True Population Mean (mu)         : %.1f ms
", true_mu))
cat(sprintf("True Population Std Dev (sigma)   : %.1f ms
", true_sigma))
cat(sprintf("Estimation Error (x_bar - mu)     : %+.3f ms
", x_bar - true_mu))
cat("==================================================
")


---

## Part 2: Point Estimation — The Mathematical Mechanics of $\bar{X}$

Why is the arithmetic sample mean $\bar{X} = \frac{1}{n}\sum_{i=1}^n X_i$ universally chosen as the point estimator for $\mu$? In frequentist theory, an estimator is evaluated by its long-run behavior under hypothetical repeated sampling.

### 2.1 Theoretical Property 1: Unbiasedness ($\mathbb{E}[\bar{X}] = \mu$)
An estimator $\hat{\theta}$ is **unbiased** if its expected value across all possible samples equals the true parameter $\theta$.
By the linearity of expectation:
$$\mathbb{E}[\bar{X}] = \mathbb{E}\left[\frac{1}{n}\sum_{i=1}^n X_i\right] = \frac{1}{n}\sum_{i=1}^n \mathbb{E}[X_i] = \frac{1}{n} \cdot (n\mu) = \mu$$
This guarantees that while any single sample mean $\bar{x}$ will miss $\mu$, the average of estimates across infinitely many samples has zero systematic bias.

### 2.2 Theoretical Property 2: Variance & Standard Error
Assuming observations $X_1, \dots, X_n$ are independent and identically distributed (i.i.d.) with variance $\sigma^2$:
$$\text{Var}(\bar{X}) = \text{Var}\left(\frac{1}{n}\sum_{i=1}^n X_i\right) = \frac{1}{n^2}\sum_{i=1}^n \text{Var}(X_i) = \frac{1}{n^2} \cdot (n\sigma^2) = \frac{\sigma^2}{n}$$
Taking the square root yields the **Standard Error of the Mean ($SE$)**:
$$SE(\bar{X}) = \frac{\sigma}{\sqrt{n}}$$

> [!IMPORTANT]
> **Deep Dive: Standard Deviation ($s$) vs. Standard Error ($SE$)**
> * **Standard Deviation ($s$)**: Quantifies the physical spread of individual data points in the population. As you collect more data ($n \to \infty$), $s$ converges to $\sigma$. It **does not shrink**.
> * **Standard Error ($SE$)**: Quantifies the precision of our estimate of $\mu$. Because of the $\sqrt{n}$ denominator, $SE \to 0$ as $n \to \infty$.
> * **The Square Root Penalty**: To cut your uncertainty in half ($SE / 2$), you must collect **$4\times$ as much data**. To reduce $SE$ by a factor of 10, you must collect **$100\times$ as much data**.

### 2.3 Theoretical Property 3: Bessel's Correction ($n-1$)
When estimating population variance $\sigma^2$ from sample data, the naive estimator $\frac{1}{n}\sum (X_i - \bar{X})^2$ is systematically biased downwards:
$$\mathbb{E}\left[\frac{1}{n}\sum_{i=1}^n (X_i - \bar{X})^2\right] = \frac{n-1}{n}\sigma^2$$
Because $\bar{X}$ is calculated from the exact same sample, the deviations $(X_i - \bar{X})$ are mechanically constrained to sum to zero ($\sum (X_i - \bar{X}) = 0$). This consumes **1 degree of freedom**. Dividing by $n-1$ (Bessel's correction) corrects the bias, producing an unbiased estimator $s^2$:
$$s^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar{X})^2 \implies \mathbb{E}[s^2] = \sigma^2$$

### 2.4 Theoretical Property 4: Consistency & Law of Large Numbers (LLN)
By Chebyshev's Inequality, for any $\epsilon > 0$:
$$P(|\bar{X}_n - \mu| \ge \epsilon) \le \frac{\text{Var}(\bar{X}_n)}{\epsilon^2} = \frac{\sigma^2}{n\epsilon^2} \xrightarrow{n \to \infty} 0$$
As sample size grows without bound, the probability that $\bar{X}$ differs from $\mu$ by even the smallest margin $\epsilon$ collapses to 0. $\bar{X}_n$ converges in probability to $\mu$ ($\bar{X}_n \xrightarrow{p} \mu$).

### 2.5 Theoretical Property 5: Efficiency, CRLB, and Maximum Likelihood (MLE)
For normally distributed data $X_i \sim \mathcal{N}(\mu, \sigma^2)$, the log-likelihood function is:
$$\ln L(\mu, \sigma^2) = -\frac{n}{2}\ln(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (X_i - \mu)^2$$
Differentiating with respect to $\mu$ and setting to zero yields the Maximum Likelihood Estimator (MLE):
$$\frac{\partial \ln L}{\partial \mu} = \frac{1}{\sigma^2}\sum_{i=1}^n (X_i - \mu) = 0 \implies \hat{\mu}_{\text{MLE}} = \frac{1}{n}\sum_{i=1}^n X_i = \bar{X}$$
The **Fisher Information** with respect to $\mu$ is:
$$I(\mu) = -\mathbb{E}\left[\frac{\partial^2 \ln L}{\partial \mu^2}\right] = \frac{n}{\sigma^2}$$
The **Cramér-Rao Lower Bound (CRLB)** states that the variance of any unbiased estimator $\hat{\mu}$ cannot be smaller than the inverse Fisher Information:
$$\text{Var}(\hat{\mu}) \ge \frac{1}{I(\mu)} = \frac{\sigma^2}{n}$$
Since $\text{Var}(\bar{X}) = \frac{\sigma^2}{n}$ exactly attains this bound, $\bar{X}$ is the **Minimum Variance Unbiased Estimator (MVUE)**. No other unbiased estimator can achieve smaller error variance.



In [ ]:
# Numerical Demonstration: Bessel's Correction via Monte Carlo
set.seed(42)
n_trials <- 25000
sample_size <- 5
true_pop_var <- 100.0

samples_matrix <- matrix(rnorm(n_trials * sample_size, mean = 0, sd = sqrt(true_pop_var)), 
                         nrow = n_trials, ncol = sample_size)
means <- rowMeans(samples_matrix)
dev_sq <- (samples_matrix - means)^2
sum_sq <- rowSums(dev_sq)

var_n <- sum_sq / sample_size
var_n_minus_1 <- sum_sq / (sample_size - 1)

cat("==================================================
")
cat("   BESSEL'S CORRECTION MONTE CARLO (N = 25,000)   
")
cat("==================================================
")
cat(sprintf("True Population Variance (sigma^2) : %.2f
", true_pop_var))
cat(sprintf("Average Naive Variance (div by n)  : %.2f (Bias = %+.2f)
", mean(var_n), mean(var_n) - true_pop_var))
cat(sprintf("Average Unbiased Var (div by n-1)  : %.2f (Bias = %+.2f)
", mean(var_n_minus_1), mean(var_n_minus_1) - true_pop_var))
cat(sprintf("Theoretical Expected Bias Factor   : (n-1)/n = %.2f
", (sample_size - 1) / sample_size))
cat("==================================================
")


---

## Part 3: The Sampling Distribution & The Central Limit Theorem

### 3.1 Known Variance vs. Unknown Variance ($Z$ vs. $T$)
To perform inference (such as building confidence intervals or hypothesis tests), we must know the **sampling distribution** of $\bar{X}$.
* **If $\sigma$ is known (rare in practice)**:
  $$Z = \frac{\bar{X} - \mu}{\sigma / \sqrt{n}} \sim \mathcal{N}(0, 1)$$
* **If $\sigma$ is unknown and estimated by $s$ (standard in practice)**:
  In 1908, William Sealy Gosset discovered that substituting the random variable $s$ for constant $\sigma$ introduces additional variability. The resulting statistic follows **Student's $t$-distribution**:
  $$T = \frac{\bar{X} - \mu}{s / \sqrt{n}} \sim t_{n-1}$$
  The $t$-distribution has heavier tails than the standard normal distribution to account for uncertainty in $s$. As $n \to \infty$, $t_{n-1} \to \mathcal{N}(0, 1)$.

### 3.2 The Central Limit Theorem (CLT)
What if the underlying population is **not normally distributed**?
The **Lindeberg-Lévy Central Limit Theorem** guarantees that if observations $X_1, X_2, \dots, X_n$ are i.i.d. with finite mean $\mu$ and finite variance $\sigma^2$, then regardless of the population distribution shape:
$$\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \xrightarrow{d} \mathcal{N}(0, 1) \quad \text{as } n \to \infty$$
Even if individual requests follow an asymmetric, skewed, or multimodal distribution, the distribution of their average $\bar{X}$ rapidly converges to a Gaussian bell curve.



In [ ]:
# Numerical Demonstration: Central Limit Theorem under Severe Skewness
set.seed(42)
lambda_rate <- 0.05
pop_mean <- 1.0 / lambda_rate
sample_sizes <- c(2, 5, 30, 100)
n_sim_clt <- 10000

clt_results <- list()
for (n_val in sample_sizes) {
    sim_matrix <- matrix(rexp(n_sim_clt * n_val, rate = lambda_rate), nrow = n_sim_clt, ncol = n_val)
    means_sim <- rowMeans(sim_matrix)
    m3 <- mean((means_sim - mean(means_sim))^3)
    s3 <- sd(means_sim)^3
    skew_val <- m3 / s3
    clt_results[[as.character(n_val)]] <- list(means = means_sim, skewness = skew_val)
}

cat("==================================================
")
cat("   CENTRAL LIMIT THEOREM: SKEWNESS DECAY (EXP)    
")
cat("==================================================
")
cat(sprintf("Population Mean: %.1f | Population Skewness: 2.00
", pop_mean))
for (n_val in sample_sizes) {
    cat(sprintf("Sample Size n = %3d | Mean(x_bar) = %5.2f | Skewness = %5.3f
", 
                n_val, mean(clt_results[[as.character(n_val)]]$means), clt_results[[as.character(n_val)]]$skewness))
}
cat("==================================================
")


---

## Part 4: Interval Estimation — Anatomy of the Frequentist Confidence Interval

A single point estimate $\bar{x} = 143.08\text{ ms}$ tells us where the estimator landed, but provides zero information about its precision. In the frequentist paradigm, uncertainty is captured by a **Confidence Interval (CI)**.

### 4.1 Mathematical Derivation of the Two-Sided $(1-\alpha)$ CI
Because $T = \frac{\bar{X} - \mu}{s/\sqrt{n}} \sim t_{n-1}$, we choose critical value $t_{\alpha/2, n-1}$ such that:
$$P\left(-t_{\alpha/2, n-1} \le \frac{\bar{X} - \mu}{s/\sqrt{n}} \le t_{\alpha/2, n-1}\right) = 1 - \alpha$$
Isolating the parameter $\mu$ inside the inequality:
$$-t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}} \le \bar{X} - \mu \le t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$
Multiplying by $-1$ and adding $\bar{X}$:
$$P\left(\bar{X} - t_{\alpha/2, n-1} \frac{s}{\sqrt{n}} \le \mu \le \bar{X} + t_{\alpha/2, n-1} \frac{s}{\sqrt{n}}\right) = 1 - \alpha$$
The **Margin of Error ($E$)** is defined as:
$$E = t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}} = t_{\alpha/2, n-1} \cdot SE$$
The resulting interval is:
$$\text{CI}_{1-\alpha} = \left[\bar{x} - E, \; \bar{x} + E\right]$$

### 4.2 The Critical Epistemological Interpretation: The 100 Universes
> [!CAUTION]
> **The Ubiquitous Confidence Interval Fallacy**
> Once you calculate an interval from your specific sample (e.g., $[134.13, 152.03]\text{ ms}$), it is **mathematically invalid** to state:
> *"There is a 95% probability that the true mean $\mu$ is between 134.13 and 152.03 ms."*
>
> In the frequentist paradigm:
> 1. $\mu$ is a **fixed constant of nature**. It does not possess a probability distribution.
> 2. The realized bounds $[134.13, 152.03]$ are fixed numbers.
> 3. Therefore, $\mu$ is either inside this specific interval or outside it. The probability is strictly **either 1 or 0**, not 0.95.
>
> **The Correct Frequentist Interpretation**:
> The 95% probability belongs to the **estimation procedure**, not the realized numbers. If we were to repeat the experiment 10,000 times, generating 10,000 independent samples and computing 10,000 confidence intervals, **95% of those calculated intervals will contain the true parameter $\mu$**, while 5% will miss it.

### 4.3 Sample Size Sizing / Margin of Error Planning
If engineering leadership requires that latency be estimated within a margin of error of at most $\pm E$ ms with $95\%$ confidence ($\alpha = 0.05$), how many samples must we collect?
$$E = z_{\alpha/2} \frac{\sigma}{\sqrt{n}} \implies \sqrt{n} = \frac{z_{\alpha/2} \sigma}{E} \implies n = \left(\frac{z_{\alpha/2} \sigma}{E}\right)^2$$



In [ ]:
# Calculate the 95% Confidence Interval for our production sample
confidence_level <- 0.95
alpha_level <- 1.0 - confidence_level
df <- n - 1

# Critical value from Student-t distribution
t_critical <- qt(1.0 - alpha_level / 2.0, df = df)
margin_of_error <- t_critical * se_mean
ci_lower <- x_bar - margin_of_error
ci_upper <- x_bar + margin_of_error

# Verify with Base R t.test
t_test_res <- t.test(sample_data, conf.level = confidence_level)

cat("==================================================
")
cat("     FREQUENTIST 95% CONFIDENCE INTERVAL          
")
cat("==================================================
")
cat(sprintf("Sample Mean (x_bar)               : %.3f ms
", x_bar))
cat(sprintf("Degrees of Freedom (df)           : %d
", df))
cat(sprintf("Critical t-value (t*)             : %.4f
", t_critical))
cat(sprintf("Margin of Error (moe)             : ±%.3f ms
", margin_of_error))
cat(sprintf("95%% Confidence Interval (Manual)  : [%.3f, %.3f] ms
", ci_lower, ci_upper))
cat(sprintf("95%% Confidence Interval (t.test)  : [%.3f, %.3f] ms
", t_test_res$conf.int[1], t_test_res$conf.int[2]))
cat(sprintf("Contains True Population Mean?    : %s
", (ci_lower <= true_mu && true_mu <= ci_upper)))
cat("==================================================
")


---

## Part 5: Beyond Normality — The Non-Parametric Bootstrap & Robust Means

What happens when production data contains severe outliers (e.g., occasional stop-the-world garbage collection pauses or network reconnect timeouts) that violate Gaussian assumptions?

### 5.1 The Empirical Non-Parametric Bootstrap (Bradley Efron, 1979)
When we cannot trust parametric assumptions, we let the data define its own sampling distribution:
1. Treat the observed sample $X = \{x_1, \dots, x_n\}$ as an empirical representation of the population ($\hat{F}_n$).
2. Draw $B = 10,000$ resamples of size $n$ **with replacement** from $X$.
3. Compute the sample mean $\bar{X}^{*(b)}$ for each resample $b \in \{1, \dots, B\}$.
4. The standard deviation of $\{\bar{X}^{*(1)}, \dots, \bar{X}^{*(B)}\}$ yields the **Bootstrap Standard Error ($SE_{\text{boot}}$)**.
5. The $2.5^{\text{th}}$ and $97.5^{\text{th}}$ percentiles yield the **95% Bootstrap Percentile Confidence Interval**.

### 5.2 The Trimmed Mean ($\bar{x}_{\alpha}$)
If extreme outliers drag the arithmetic mean, the **trimmed mean** discards a fixed fraction $\alpha$ (e.g., $10\%$) from each tail of the sorted data:
$$\bar{x}_{\alpha} = \frac{1}{n - 2k}\sum_{i=k+1}^{n-k} X_{(i)} \quad \text{where } k = \lfloor n \alpha \rfloor$$
The trimmed mean provides a robust middle-ground between the sensitive sample mean (0% breakdown point) and the median (50% breakdown point).



In [ ]:
# Non-Parametric Bootstrap and Trimmed Mean Implementation
set.seed(42)
B <- 10000
boot_matrix <- matrix(sample(sample_data, size = B * n, replace = TRUE), nrow = B, ncol = n)
boot_means <- rowMeans(boot_matrix)

boot_ci_lower <- quantile(boot_means, probs = 0.025)
boot_ci_upper <- quantile(boot_means, probs = 0.975)
boot_se <- sd(boot_means)
boot_mean_val <- mean(boot_means)

# Trimmed Means (10% and 20%)
trimmed_mean_10 <- mean(sample_data, trim = 0.10)
trimmed_mean_20 <- mean(sample_data, trim = 0.20)

cat("==================================================
")
cat("    BOOTSTRAP & ROBUST ESTIMATION RESULTS         
")
cat("==================================================
")
cat(sprintf("Original Sample Mean              : %.3f ms
", x_bar))
cat(sprintf("Bootstrap Mean (B = %d)       : %.3f ms
", B, boot_mean_val))
cat(sprintf("Bootstrap Standard Error          : %.3f ms
", boot_se))
cat(sprintf("Bootstrap 95%% Percentile CI       : [%.3f, %.3f] ms
", boot_ci_lower, boot_ci_upper))
cat(sprintf("10%% Trimmed Mean                  : %.3f ms
", trimmed_mean_10))
cat(sprintf("20%% Trimmed Mean                  : %.3f ms
", trimmed_mean_20))
cat("==================================================
")


---

## Part 6: Comprehensive Multi-Panel Visualization Dashboard

We now synthesize all 4 key frequentist perspectives into a publication-grade diagnostic dashboard ($15 \times 11$, 120 DPI):
1. **Raw Observations vs. Sampling Distribution**: Visualizes the dramatic difference between data spread ($s$) and estimator precision ($SE$).
2. **The 100 Parallel Universes of Confidence Intervals**: 100 simulated benchmark trials demonstrating that exactly ~95% of computed intervals cover true $\mu$ (green) while ~5% miss (red).
3. **The Central Limit Theorem Under Heavy Skew**: Overlaid sampling distributions showing the rapid transformation of exponential data into Gaussian bell curves as $n$ scales.
4. **Parametric Student-$t$ vs. Non-Parametric Bootstrap CI**: Visualizing empirical bootstrap replicates alongside parametric interval bounds.



In [ ]:
# Multi-Panel Visualization Dashboard in Base R
par(mfrow = c(2, 2), mar = c(4.5, 4.5, 3.5, 1.5))

# Panel 1: Raw Sample Latency Distribution
hist(sample_data, breaks = 15, prob = TRUE, col = "#e6f2ff", border = "#1f77b4",
     main = "1. Observed Sample Distribution (n=50)", xlab = "Latency (ms)", ylab = "Density", cex.main = 1.1)
lines(density(sample_data), col = "#1f77b4", lwd = 2)
abline(v = x_bar, col = "#1f77b4", lwd = 2.5, lty = 1)
abline(v = true_mu, col = "black", lwd = 2.5, lty = 2)
legend("topright", legend = c(sprintf("Sample Mean = %.1f ms", x_bar), sprintf("True mu = %.1f ms", true_mu)),
       col = c("#1f77b4", "black"), lty = c(1, 2), lwd = 2, bty = "n", cex = 0.85)

# Panel 2: The 100 Parallel Universes
set.seed(42)
n_univ <- 100
plot(0, 0, type = "n", xlim = c(130, 160), ylim = c(1, n_univ),
     main = "2. The 100 Parallel Universes (95% CIs)", xlab = "Confidence Interval (ms)", ylab = "Experiment Index", cex.main = 1.1)
abline(v = true_mu, col = "black", lwd = 2, lty = 2)

covered_cnt <- 0
for (idx in 1:n_univ) {
    u_samp <- rnorm(n, mean = true_mu, sd = true_sigma)
    u_m <- mean(u_samp); u_se <- sd(u_samp) / sqrt(n)
    u_low <- u_m - qt(0.975, df = n - 1) * u_se
    u_high <- u_m + qt(0.975, df = n - 1) * u_se
    is_cov <- (u_low <= true_mu && true_mu <= u_high)
    if (is_cov) covered_cnt <- covered_cnt + 1
    col_line <- if (is_cov) "#2ca02c" else "#d62728"
    lines(c(u_low, u_high), c(idx, idx), col = col_line, lwd = 1.5)
    points(u_m, idx, col = col_line, pch = 20, cex = 0.6)
}
legend("topright", legend = sprintf("Coverage: %d%%", covered_cnt), bty = "n", cex = 0.9)

# Panel 3: Central Limit Theorem
plot(density(clt_results[["100"]]$means), col = "#1f77b4", lwd = 2, xlim = c(5, 45),
     main = "3. Central Limit Theorem: Skew Decay", xlab = "Sample Mean (x_bar)", ylab = "Density", cex.main = 1.1)
lines(density(clt_results[["30"]]$means), col = "#2ca02c", lwd = 2)
lines(density(clt_results[["5"]]$means), col = "#9467bd", lwd = 2)
lines(density(clt_results[["2"]]$means), col = "#ff7f0e", lwd = 2)
abline(v = pop_mean, col = "black", lty = 3, lwd = 2)
legend("topright", legend = c("n=2", "n=5", "n=30", "n=100"), col = c("#ff7f0e", "#9467bd", "#2ca02c", "#1f77b4"),
       lty = 1, lwd = 2, bty = "n", cex = 0.85)

# Panel 4: Parametric vs Bootstrap CI
hist(boot_means, breaks = 40, prob = TRUE, col = "#e0f7fa", border = "#00838f",
     main = "4. Parametric Student-t vs Bootstrap CI", xlab = "Mean Latency (ms)", ylab = "Density", cex.main = 1.1)
x_grid <- seq(135, 155, length.out = 300)
lines(x_grid, dt((x_grid - x_bar) / se_mean, df = df) / se_mean, col = "#d62728", lwd = 2, lty = 2)
lines(c(ci_lower, ci_upper), c(0.015, 0.015), col = "#d62728", lwd = 5)
lines(c(boot_ci_lower, boot_ci_upper), c(0.030, 0.030), col = "#1f77b4", lwd = 5)
abline(v = true_mu, col = "black", lty = 3, lwd = 2)
legend("topright", legend = c("Bootstrap Density", "Parametric Student-t", "95% Student-t CI", "95% Bootstrap CI"),
       col = c("#00838f", "#d62728", "#d62728", "#1f77b4"), lty = c(1, 2, 1, 1), lwd = c(2, 2, 4, 4), bty = "n", cex = 0.75)


---

## Part 7: Duality Between Confidence Intervals & Hypothesis Testing

In frequentist statistics, **interval estimation** and **hypothesis testing** are two sides of the exact same mathematical coin.

### 7.1 The One-Sample $t$-Test
Suppose a Service Level Agreement (SLA) requires that the mean processing latency be no more than $\mu_0$ ms.
We test:
* **Null Hypothesis ($H_0$)**: $\mu = \mu_0$
* **Alternative Hypothesis ($H_1$)**: $\mu \neq \mu_0$ (Two-sided test)

The test statistic measures how many standard errors the sample mean $\bar{x}$ lies away from the hypothesized value $\mu_0$:
$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}} = \frac{\bar{x} - \mu_0}{SE}$$
Under $H_0$, this statistic follows Student's $t$-distribution with $n - 1$ degrees of freedom. The two-sided $p$-value is:
$$p = 2 \cdot P(T \ge |t|)$$

### 7.2 The Inversion Duality Theorem
> [!IMPORTANT]
> **The Confidence Interval / Hypothesis Test Equivalence**
> A two-sided $(1-\alpha)$ Confidence Interval consists of **all hypothesized values $\mu_0$ that fail to be rejected** at significance level $\alpha$.
> * If a candidate SLA value $\mu_0$ falls **inside** the 95% Confidence Interval $[\text{CI}_{\text{low}}, \text{CI}_{\text{high}}] \iff p \ge 0.05$ (Fail to reject $H_0$).
> * If a candidate SLA value $\mu_0$ falls **outside** the 95% Confidence Interval $[\text{CI}_{\text{low}}, \text{CI}_{\text{high}}] \iff p < 0.05$ (Reject $H_0$).



In [ ]:
# Demonstration of Duality: Testing two candidate null values mu_0
mu_candidates <- c(140.0, 160.0)

cat("========================================================================
")
cat(sprintf("  DUALITY DEMONSTRATION: Observed 95%% CI = [%.2f, %.2f] ms
", ci_lower, ci_upper))
cat("========================================================================
")
cat(sprintf("%-16s | %-10s | %-12s | %-16s | %s
", "Null Value (mu0)", "t-stat", "p-value", "Inside 95% CI?", "Decision (alpha=0.05)"))
cat("------------------------------------------------------------------------
")

for (mu_0 in mu_candidates) {
    t_res <- t.test(sample_data, mu = mu_0)
    inside_ci <- (ci_lower <= mu_0 && mu_0 <= ci_upper)
    decision <- if (t_res$p.value >= 0.05) "Fail to Reject H0" else "Reject H0 (p < 0.05)"
    cat(sprintf("mu0 = %-10.1f | %-10.3f | %-12.4e | %-16s | %s
", 
                mu_0, t_res$statistic, t_res$p.value, as.character(inside_ci), decision))
}
cat("========================================================================
")


---

## 🎯 Check Your Understanding: 3-Tier Scaffolding Exercises

### 🟢 Tier 1: Conceptual Verification
1. **Scenario A**: An intern reports: *"We benchmarked 20 database queries. The sample standard deviation was $s = 14\text{ ms}$, so the 95% Confidence Interval has width $\pm 14\text{ ms}$."* What error did the intern make?
2. **Scenario B**: A production pipeline currently measures latency with a standard error of $SE = 4.0\text{ ms}$ based on $n = 50$ requests. Product management demands that the standard error be cut down to $SE = 1.0\text{ ms}$. How many total requests $N$ must you collect?
3. **Scenario C**: If you compute a 95% confidence interval of $[88.2, 94.6]\text{ ms}$ from a sample, can you conclude that there is a 95% chance that the next single request will take between 88.2 and 94.6 ms?

<details>
<summary><b>🔍 Click here to view the Tier 1 answers</b></summary>

1. **Answer A**: The intern confused the **sample standard deviation ($s$)** with the **margin of error ($E$)**. The standard deviation measures individual query dispersion. The margin of error is $t^* \cdot \frac{s}{\sqrt{n}} \approx 2.093 \cdot \frac{14}{\sqrt{20}} \approx \pm 6.55\text{ ms}$, not $\pm 14\text{ ms}$.
2. **Answer B**: Precision scales with $\sqrt{n}$. To reduce $SE$ by a factor of 4 (from 4.0 to 1.0), sample size must increase by a factor of $4^2 = 16$. Therefore, $N = 50 \times 16 = 800$ requests.
3. **Answer C**: **Absolutely not.** A Confidence Interval is for the **population mean $\mu$**, not for individual future observations. Individual observations vary according to $\sigma \approx s$, which is $\sqrt{n}$ times wider. An interval for individual future observations is called a **Prediction Interval**: $[\bar{x} \pm t^* s \sqrt{1 + 1/n}]$.
</details>



In [ ]:
# ============================================================================
# 🟡 Tier 2: Guided Calculation — Implement `frequentist_mean_estimate()`
# ============================================================================

frequentist_mean_estimate <- function(data, confidence = 0.95) {
    n_obs <- length(data)
    mean_val <- sum(data) / n_obs
    variance_val <- sum((data - mean_val)^2) / (n_obs - 1)
    s_val <- sqrt(variance_val)
    se_val <- s_val / sqrt(n_obs)
    df_val <- n_obs - 1
    alpha_val <- 1.0 - confidence
    t_critical <- qt(1.0 - alpha_val / 2.0, df = df_val)
    moe_val <- t_critical * se_val
    lower_bound <- mean_val - moe_val
    upper_bound <- mean_val + moe_val
    
    return(list(
        n = n_obs,
        mean = mean_val,
        s = s_val,
        se = se_val,
        df = df_val,
        t_crit = t_critical,
        moe = moe_val,
        ci_lower = lower_bound,
        ci_upper = upper_bound
    ))
}

# Execute on our sample data and verify
test_res <- frequentist_mean_estimate(sample_data, confidence = 0.95)
cat("Tier 2 Implementation Test:
")
cat(sprintf("  n = %d, Mean = %.2f, SE = %.2f
", test_res$n, test_res$mean, test_res$se))
cat(sprintf("  95%% CI = [%.2f, %.2f] ms
", test_res$ci_lower, test_res$ci_upper))

# Unit test assertions
stopifnot(abs(test_res$mean - x_bar) < 1e-6)
stopifnot(abs(test_res$s - s_unbiased) < 1e-6)
stopifnot(abs(test_res$ci_lower - ci_lower) < 1e-6)
stopifnot(abs(test_res$ci_upper - ci_upper) < 1e-6)
cat("✅ All Tier 2 assertions passed successfully!
")


<details>
<summary><b>🔍 Click here to view the Tier 2 detailed line-by-line explanation</b></summary>

```python
# The function implements the exact classical derivation:
# 1. Sample mean: sum(x) / n
# 2. Bessel-corrected variance: sum((x - mean)^2) / (n - 1)
# 3. Standard error: s / sqrt(n)
# 4. Critical t-value: stats.t.ppf(1 - alpha/2, df = n - 1)
# 5. Bounds: mean ± t* · SE
```
</details>



---

### 🔴 Tier 3: Production Simulation Challenge — The Heavy-Tailed Latency Trap
> **The Problem**: Real-world cloud service request latencies are frequently **log-normally distributed** or heavy-tailed. When skewness is severe and sample sizes are small ($n = 25$), the standard Student-$t$ confidence interval suffers from **coverage breakdown**—it under-covers, failing to attain the nominal 95% rate because the sampling distribution has not yet converged to normality.
>
> **Your Task**:
> 1. Simulate 2,000 independent samples of size $n = 25$ from a Log-Normal distribution with $\mu_{\log} = 4.0, \sigma_{\log} = 0.85$ (True population mean $\mu = e^{\mu_{\log} + \sigma_{\log}^2 / 2} \approx 78.36\text{ ms}$).
> 2. For each sample, compute:
>    * **Method A**: The naive standard Student-$t$ 95% Confidence Interval on raw data.
>    * **Method B**: The Non-Parametric Bootstrap Percentile 95% CI.
> 3. Measure the empirical coverage percentage of each method against the true mean $\mu$.



In [ ]:
# Production Simulation: The Heavy-Tailed Latency Trap
set.seed(42)

log_mu <- 4.0
log_sigma <- 0.85
true_lognormal_mean <- exp(log_mu + 0.5 * (log_sigma^2)) # ~78.36 ms

n_sim <- 2000
n_obs_tier3 <- 25
B_boot <- 500

t_covered <- 0
boot_covered <- 0

cat("Running Monte Carlo simulation (2,000 replications)...
")
for (sim in 1:n_sim) {
    y_raw <- rlnorm(n_obs_tier3, meanlog = log_mu, sdlog = log_sigma)
    
    # Method A: Naive Student-t CI
    m_raw <- mean(y_raw)
    se_raw <- sd(y_raw) / sqrt(n_obs_tier3)
    moe_raw <- qt(0.975, df = n_obs_tier3 - 1) * se_raw
    if (m_raw - moe_raw <= true_lognormal_mean && true_lognormal_mean <= m_raw + moe_raw) {
        t_covered <- t_covered + 1
    }
    
    # Method B: Bootstrap Percentile CI (subset of 500 runs)
    if (sim <= 500) {
        boot_samp <- matrix(sample(y_raw, size = B_boot * n_obs_tier3, replace = TRUE), nrow = B_boot, ncol = n_obs_tier3)
        b_means <- rowMeans(boot_samp)
        b_low <- quantile(b_means, probs = 0.025)
        b_high <- quantile(b_means, probs = 0.975)
        if (b_low <= true_lognormal_mean && true_lognormal_mean <= b_high) {
            boot_covered <- boot_covered + 1
        }
    }
}

cat("====================================================================
")
cat("  TIER 3 SIMULATION RESULTS: HEAVY-TAILED COVERAGE BREAKDOWN
")
cat("====================================================================
")
cat(sprintf("True Population Mean Latency      : %.2f ms
", true_lognormal_mean))
cat(sprintf("Naive Student-t Empirical Coverage: %.1f%% (Nominal Target: 95.0%%)
", 100 * t_covered / n_sim))
cat(sprintf("Bootstrap Empirical Coverage      : %.1f%%
", 100 * boot_covered / 500))
cat("====================================================================
")
cat("Takeaway: When data is skewed and n is small, naive Student-t intervals
")
cat("suffer from systematic under-coverage due to slow asymptotic CLT convergence.
")


<details>
<summary><b>🔍 Click here to view the Tier 3 solution & architectural takeaway</b></summary>

### The Takeaway for Production Systems:
1. **Asymmetric Failure**: When underlying metrics are positively skewed (typical of request latencies), the sample mean is correlated with the sample standard deviation. Outliers in a small sample inflate both $\bar{x}$ and $s$, leading to intervals that fail to cover the true parameter primarily on the left tail.
2. **Log-Transformation**: For strictly positive, right-skewed data, log-transforming ($Y = \ln(X)$) normalizes the data, allowing standard Gaussian interval construction before mapping back to physical scale via the log-normal expectation $\mu = \exp(\mu_Y + \sigma_Y^2 / 2)$.
3. **Bootstrap Calibration**: When no distribution can be assumed, Studentized Bootstrap (Bootstrap-$t$) or BCa (Bias-Corrected and Accelerated) intervals correct for skewness and restore 95% nominal coverage.
</details>



---

## 🏁 Summary Checklist: How to Calculate a Mean in Practice

```
               Is your sample size n >= 30 and data roughly symmetric?
                                   /             \
                                  /               \
                               [YES]             [NO]
                                 |                 |
                   Use Classical Student-t       Are there extreme outliers or
                       Confidence Interval       severe positive skew (e.g. latency)?
                                                   /                   \
                                                  /                     \
                                            [Outliers]               [Skewed]
                                                |                        |
                                        Use Trimmed Mean          Use Log-Transform
                                      or Non-Parametric          or Non-Parametric
                                          Bootstrap                   Bootstrap
```

### Essential Frequentist Formula Card:
* **Sample Mean**: $\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$
* **Bessel-Corrected Sample Variance**: $s^2 = \frac{1}{n-1}\sum_{i=1}^n (x_i - \bar{x})^2$
* **Standard Error of the Mean**: $SE = \frac{s}{\sqrt{n}}$
* **Two-Sided $(1-\alpha)$ Confidence Interval**: $\left[\bar{x} - t_{\alpha/2, n-1} \cdot SE, \; \bar{x} + t_{\alpha/2, n-1} \cdot SE\right]$
* **Required Sample Size for Margin of Error $E$**: $n = \left(\frac{z_{\alpha/2} \cdot s}{E}\right)^2$
* **One-Sample $t$-Statistic for SLA Check $\mu_0$**: $t = \frac{\bar{x} - \mu_0}{SE}, \quad df = n - 1$

---
**[🏠 Return to Course Home](README.md) | [📊 Next Sheet: Two-Sample Comparison (t-Test)](sheet_t_test_frequentist_vs_bayesian.ipynb)**
